# Align

This notebook reads tabular projections from `3-project`, builds positive same-document model pairs, runs alignment algorithms across projection slices in parallel, and persists all candidate alignments to `artifacts/align/alignment-candidates.csv`.

In [2]:
import os
from importlib.metadata import version

import pandas as pd
from IPython.display import JSON, display

from src.lib.align_pipeline import (
    align_run_paths,
    append_alignment_candidates,
    available_cpu_count,
    bdikit_method_names,
    build_alignment_tasks,
    build_positive_pairs,
    iter_alignment_task_results_parallel,
    load_alignment_candidates,
    load_project_artifacts,
    magneto_method_names,
    valentine_method_names,
    write_pairs_csv,
)
from src.lib.infer_pipeline import find_repo_root, parse_optional_int

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
repo_root = find_repo_root()
display(
    JSON(
        {
            "repo_root": repo_root.as_posix(),
            "valentine": version("valentine"),
            "bdi-kit": version("bdi-kit"),
            "magneto-python": version("magneto-python"),
            "torch": version("torch"),
        },
        expanded=True,
    )
)

<IPython.core.display.JSON object>

In [ ]:
model_index, elements = load_project_artifacts(repo_root)
pairs = build_positive_pairs(model_index)
max_pairs = parse_optional_int(os.getenv("ALIGN_MAX_PAIRS"))
match_timeout_seconds = parse_optional_int(os.getenv("ALIGN_MATCH_TIMEOUT_SECONDS"))
if match_timeout_seconds is None:
    match_timeout_seconds = 240
worker_count = available_cpu_count()
if max_pairs is not None:
    pairs = pairs.head(max_pairs).copy()

paths = align_run_paths(repo_root)
if paths.candidates_csv.exists():
    paths.candidates_csv.unlink()
write_pairs_csv(repo_root, pairs)

projection_specs = [
    ("property", "path_only"),
    ("property", "path_plus_metadata_values"),
    ("type", "members_as_values"),
    ("relation", "path_plus_metadata_values"),
]

tasks = build_alignment_tasks(
    repo_root,
    pairs,
    projection_specs=projection_specs,
)

pair_breakdown = (
    pairs.groupby(["source_document_id", "source_condition", "target_condition"])
    .size()
    .to_frame("pair_count")
    .reset_index()
)

snapshot = {
    "pair_count": int(len(pairs)),
    "projection_specs": [f"{layer}/{mode}" for layer, mode in projection_specs],
    "task_count": int(len(tasks)),
    "valentine_methods": valentine_method_names(),
    "bdikit_methods": bdikit_method_names(),
    "magneto_methods": magneto_method_names(),
    "max_pairs": max_pairs,
    "match_timeout_seconds": match_timeout_seconds,
    "worker_count": worker_count,
    "pairs_csv": paths.pairs_csv.relative_to(repo_root).as_posix(),
    "candidates_csv": paths.candidates_csv.relative_to(repo_root).as_posix(),
    "scenarios": sorted(model_index["condition"].astype(str).unique().tolist()),
}
display(JSON(snapshot, expanded=True))
display(pair_breakdown)

In [ ]:
print(
    f"Starting parallel align for {len(pairs)} pair(s), {len(tasks)} task(s), "
    f"with {worker_count} worker(s); per-method timeout = {match_timeout_seconds}s",
    flush=True,
)

append_alignment_candidates(repo_root, [], append=False)

written_rows = 0
failure_rows = []
completed_tasks = 0

for result in iter_alignment_task_results_parallel(
    repo_root,
    tasks,
    timeout_seconds=match_timeout_seconds,
    max_workers=worker_count,
):
    completed_tasks += 1
    task_rows = result["rows"]
    task_failures = result["failures"]
    if task_rows:
        append_alignment_candidates(repo_root, task_rows, append=True)
        written_rows += len(task_rows)
    if task_failures:
        failure_rows.extend(task_failures)
    print(
        f"[task {completed_tasks}/{len(tasks)} | pair #{result['pair_index']}] "
        f"{result['backend']}/{result['method']} "
        f"{result['projection_layer']}/{result['projection_mode']} "
        f"rows={len(task_rows)} failures={len(task_failures)} "
        f"pair={result['pair_id']}",
        flush=True,
    )

failures_df = pd.DataFrame(failure_rows)
display(
    {
        "written_rows": written_rows,
        "failure_count": int(len(failures_df)),
        "pair_count": int(len(pairs)),
        "task_count": int(len(tasks)),
        "worker_count": worker_count,
    }
)
if not failures_df.empty:
    display(failures_df)

In [ ]:
if not paths.candidates_csv.exists():
    raise FileNotFoundError(f"Missing {paths.candidates_csv}")

candidates = load_alignment_candidates(repo_root)
candidate_summary = (
    candidates.groupby(
        [
            "projection_layer",
            "projection_mode",
            "backend",
            "method",
            "source_condition",
            "target_condition",
        ]
    )
    .agg(
        pair_count=("pair_id", "nunique"),
        candidate_count=("source_element_id", "count"),
        mean_score=("score", "mean"),
    )
    .reset_index()
    .sort_values(
        [
            "projection_layer",
            "projection_mode",
            "backend",
            "method",
            "source_condition",
            "target_condition",
        ],
        ignore_index=True,
    )
)

display(candidates.head(20))
display(candidate_summary)